<a href="https://colab.research.google.com/github/3OMDEH/Machine-Learning-Projects-Beginner-/blob/main/FraudDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Importing Libraries**

In [ ]:
#@title Dataset Link: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud

In [ ]:
# Core dependencies: %pip install pandas numpy matplotlib scikit-learn
# Optional: %pip install kagglehub imbalanced-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (roc_auc_score, RocCurveDisplay, PrecisionRecallDisplay,
    precision_recall_curve, average_precision_score, precision_score, f1_score,
    recall_score, confusion_matrix, classification_report, ConfusionMatrixDisplay, log_loss)
RANDOM_STATE = 42
DATA_PATH = Path('creditcard.csv')
if not DATA_PATH.exists():
    try:
        import kagglehub
    except ImportError as error:
        raise ImportError('Place creditcard.csv beside this notebook or install kagglehub.') from error
    path = Path(kagglehub.dataset_download('mlg-ulb/creditcardfraud'))
    DATA_PATH = path / 'creditcard.csv'  # Explicit file inside the downloaded directory

# **Loading Dataset**

In [ ]:
# pd.read_csv(path) does not work when path is an entire directory, not a CSV file.
df = pd.read_csv(DATA_PATH)
feature_columns = ['Time'] + [f'V{i}' for i in range(1, 29)] + ['Amount']
required_columns = feature_columns + ['Class']
if not set(required_columns).issubset(df.columns):
    raise ValueError(f'Expected columns: {required_columns}')
df = df[required_columns].copy()
if df['Class'].isna().any() or not df['Class'].isin([0, 1]).all():
    raise ValueError('Class must contain observed binary labels 0 and 1.')
if not all(pd.api.types.is_numeric_dtype(df[col]) for col in feature_columns):
    raise ValueError('Features must be numeric.')
if df['Time'].isna().any():
    raise ValueError('Time is required for chronological splitting.')
if np.isinf(df[feature_columns].to_numpy()).any():
    raise ValueError('Infinite feature values need investigation before training.')
print('Loaded:', DATA_PATH, df.shape)

# **Data Exploration**

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
#@title Checking for missing values
df.isnull().sum()

In [ ]:
# Showing correlation between features: descriptive inspection only.
# Do not choose features using target correlations from the future test period.
# A training-only correlation table is shown after splitting below.

In [ ]:
# Pie Chart: use actual class indices rather than hard-coded frequency order.
class_counts = df['Class'].value_counts().sort_index()
print('Class counts:', class_counts.to_dict())
print(f"Fraud prevalence: {df['Class'].mean():.4%}")
plt.figure(figsize=(8, 4))
plt.pie(class_counts, labels=[{0: 'Non-Fraud', 1: 'Fraud'}[i] for i in class_counts.index],
        autopct='%1.2f%%')
plt.title('Class distribution')
plt.show()

# **Data Fixing**

In [ ]:
# Exact-record deduplication is an explicit benchmark choice inherited from this project.
# Identical records may be legitimate transactions; audit this choice with transaction IDs
# on real data. No IDs here means we cannot prove these represent duplicate events.
print('Exact duplicate records removed:', int(df.duplicated().sum()))
df = df.drop_duplicates().sort_values('Time', kind='stable').reset_index(drop=True)
print('Rows retained:', len(df))

V1–V28 are supplied PCA features; that does not guarantee equal scales. We keep them as supplied for this logistic-regression baseline and scale Time/Amount with training-only statistics. Robust scaling reduces sensitivity to extreme values; it does not remove skewness. `np.log1p(Amount)` is a possible later comparison and handles zero amounts. Imbalance, temporal drift, threshold selection, and probability calibration all need attention.

# **Data Splitting**

In [ ]:
# Splitting into x and y: train on earlier transactions, evaluate on later ones.
# Keep transactions with identical Time in the same period.
unique_times = np.sort(df['Time'].unique())
if len(unique_times) < 5:
    raise ValueError('Too few distinct transaction times for three chronological periods.')
train_cutoff = unique_times[int(len(unique_times) * 0.6)]
dev_cutoff = unique_times[int(len(unique_times) * 0.8)]
train_mask = df['Time'] < train_cutoff
dev_mask = (df['Time'] >= train_cutoff) & (df['Time'] < dev_cutoff)
test_mask = df['Time'] >= dev_cutoff
x = df[feature_columns].copy()
y = df['Class'].astype(int)
x_train, y_train = x.loc[train_mask], y.loc[train_mask]
x_dev, y_dev = x.loc[dev_mask], y.loc[dev_mask]
x_test, y_test = x.loc[test_mask], y.loc[test_mask]
for name, features, labels in [('train', x_train, y_train),
                              ('development', x_dev, y_dev), ('test', x_test, y_test)]:
    if labels.nunique() != 2:
        raise ValueError(f'{name} needs both classes for the planned metrics; inspect date boundaries.')
    assert features.index.equals(labels.index)
    print(name, len(labels), 'transactions;', int(labels.sum()), 'frauds')
assert x_train.Time.max() < x_dev.Time.min()
assert x_dev.Time.max() < x_test.Time.min()
# These are approximate 60/20/20 periods by distinct timestamps, not exact row fractions.
display(pd.concat([x_train, y_train], axis=1).corr()['Class'].sort_values())

In [ ]:
# Scaling Time and Amount inside a complete pipeline prevents refitting on test data.
# Imputation is fitted on training only if future versions contain missing values.
scaled_columns = ['Time', 'Amount']
pca_columns = [f'V{i}' for i in range(1, 29)]
scaled_preprocessing = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
    ('scaler', RobustScaler()),
])
preprocessor = ColumnTransformer([
    ('time_amount', scaled_preprocessing, scaled_columns),
    ('pca', SimpleImputer(strategy='median', keep_empty_features=True), pca_columns),
])

In [ ]:
# Amount scaling is included above: no manual mutation of x_train or x_test.
# Each candidate fits a separate clone of preprocessor on training rows only.

In [ ]:
y_train.value_counts()

In [ ]:
# Handling Class Imbalance: compare no weighting and class weighting first.
# Optional SMOTE comparison requires imbalanced-learn and additional memory.
USE_SMOTE = False
if USE_SMOTE:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ResamplingPipeline
    if y_train.value_counts().min() < 6:
        raise ValueError('Default SMOTE requires at least six minority training samples.')
# Do not overwrite original training data or resample development/test transactions.
# If using SMOTE, compare it separately from class_weight, rather than combining blindly.

RobustScaler is a reasonable starting choice for Time/Amount because it uses the median and interquartile range. Fit it on training data only. Zero amounts are valid; `log1p`, unlike plain `log`, also handles zero. SMOTE uses distances across all features, so scaling only Time/Amount is a baseline rather than proof that all SMOTE distances are well balanced.

# **Building the model**

In [ ]:
# Building the model: select using development average precision (AP), not accuracy.
candidates = {
    'Prior baseline': Pipeline([('preprocessor', clone(preprocessor)),
                               ('model', DummyClassifier(strategy='prior'))]),
    'Logistic regression': Pipeline([('preprocessor', clone(preprocessor)),
        ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))]),
    'Weighted logistic regression': Pipeline([('preprocessor', clone(preprocessor)),
        ('model', LogisticRegression(class_weight='balanced', max_iter=2000,
                                     random_state=RANDOM_STATE))]),
}
if USE_SMOTE:
    candidates['SMOTE logistic regression'] = ResamplingPipeline([
        ('preprocessor', clone(preprocessor)),
        # Partial oversampling avoids expanding fraud to half of a large dataset.
        ('sampler', SMOTE(sampling_strategy=0.1, random_state=RANDOM_STATE)),
        ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ])

def fraud_scores(model, features):
    positive_index = list(model.classes_).index(1)
    return model.predict_proba(features)[:, positive_index]

rows = []
for name, candidate in candidates.items():
    candidate.fit(x_train, y_train)
    scores = fraud_scores(candidate, x_dev)
    rows.append({'model': name, 'dev_average_precision': average_precision_score(y_dev, scores),
                 'dev_roc_auc': roc_auc_score(y_dev, scores)})
comparison = pd.DataFrame(rows).sort_values('dev_average_precision', ascending=False)
display(comparison)
best_name = comparison.iloc[0]['model']
LR_model = candidates[best_name]  # Preserve the familiar variable; winner may be baseline.
print('Selected:', best_name)

# Choose an illustrative F1-maximizing alert threshold on DEVELOPMENT data only.
# A production policy should instead specify review capacity or error costs.
dev_scores = fraud_scores(LR_model, x_dev)
precision, recall, thresholds = precision_recall_curve(y_dev, dev_scores)
f1_values = np.divide(2 * precision[:-1] * recall[:-1],
                      precision[:-1] + recall[:-1],
                      out=np.zeros_like(thresholds),
                      where=(precision[:-1] + recall[:-1]) > 0)
# Last maximum favors the higher threshold in an exact tie.
best_threshold_index = np.flatnonzero(f1_values == f1_values.max())[-1]
alert_threshold = float(thresholds[best_threshold_index])
print(f'Chosen development threshold: {alert_threshold:.6f}')
# Keep the trained model fixed: refitting it could change the score scale and threshold.
# Do not report development-selected F1 as independent test performance.

In [ ]:
# Final evaluation: run once after the model and threshold are chosen.
y_true = y_test
y_pred_proba = fraud_scores(LR_model, x_test)
y_pred = (y_pred_proba >= alert_threshold).astype(int)
print('Final model:', best_name, '| threshold:', alert_threshold)
print('Test average precision:', average_precision_score(y_true, y_pred_proba))
print('Test fraud prevalence (constant-score AP baseline):', float(y_true.mean()))
print('Test ROC AUC:', roc_auc_score(y_true, y_pred_proba))
print('Test log loss:', log_loss(y_true, np.column_stack([1-y_pred_proba, y_pred_proba]), labels=[0, 1]))
print(classification_report(y_true, y_pred, labels=[0, 1],
      target_names=['Non-Fraud', 'Fraud'], zero_division=0))
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
print(f'Frauds found: {tp}; missed: {fn}; false alerts: {fp}; total alerts: {tp + fp}')
print(f'Alerts per 1,000 transactions: {1000 * (tp + fp) / len(y_true):.2f}')
print(f'False-positive rate: {fp / (fp + tn):.6f}')
ConfusionMatrixDisplay(cm, display_labels=['Non-Fraud', 'Fraud']).plot(cmap='Blues')
plt.show()

# ROC Curve: use an explicit axis to avoid empty extra figures.
fig, ax = plt.subplots(figsize=(8, 6))
RocCurveDisplay.from_predictions(y_true, y_pred_proba, ax=ax)
ax.set_title('Final test ROC curve')
ax.grid(True)
plt.show()

In [ ]:
# Precision-Recall Curve: especially informative when fraud is rare.
fig, ax = plt.subplots(figsize=(8, 6))
PrecisionRecallDisplay.from_predictions(y_true, y_pred_proba, ax=ax)
ax.axhline(y_true.mean(), color='gray', linestyle='--', label='Fraud prevalence')
ax.set_title('Final test precision-recall curve')
ax.legend()
ax.grid(True)
plt.show()
# Average precision summarizes ranking; thresholded precision/recall describe alert behavior.
# Reweighting/SMOTE can alter calibration: scores are not verified fraud probabilities.

## Pipeline review and practical limits
- Previously, Time and Amount each called `fit_transform` on the test set. That learned test-specific scales and made train/test representations inconsistent. Each complete pipeline now learns its imputer/scaler on training rows and only transforms later periods.
- SMOTE was applied to training only, which was correct. After exact balancing, `class_weight='balanced'` becomes equal weights, so the old combination was redundant. The optional comparison now uses resampling separately.
- Chronological splitting asks whether earlier transactions predict later ones and keeps equal timestamps together. It cannot address cardholder grouping without IDs or label delays without label-availability timestamps. A real deployment may require a training gap.
- Development data selects both the ranking model and an illustrative F1 threshold. That makes development performance optimistic; test data remains separate. With more data, use distinct model-selection and threshold-validation periods.
- F1 assumes a particular precision/recall tradeoff. Real fraud review needs an explicit alert budget or false-alert/missed-fraud costs. Transaction Amount alone does not define those costs.
- The prior baseline makes the weakness of accuracy clear: predicting no fraud can look accurate while finding none. Compare AP alongside recall, precision, false alerts, and alert volume.
- The supplied V features are already anonymized PCA outputs. We cannot audit the original transformation or reconstruct its fit using this CSV. Keeping PCA outputs does not imply unit variance.
- Exact-row deduplication preserves the original project's choice. Without transaction IDs, identical rows are not proof of duplicated events; inspect this choice before using it on real logs.
- Old test results and the new chronological results are not directly comparable. This benchmark was already inspected previously; fresh external transactions are needed for a genuinely new estimate.

References: [scikit-learn threshold selection](https://scikit-learn.org/stable/modules/classification_threshold.html), [dataset source](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud).

## Original comments and learning notes
Changed original cells are preserved below for reference. The executable cells above contain the corrections.

<details>
<summary>Original cell 4</summary>

```text
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE # Used to solve the Imbalance problem
from sklearn.metrics import roc_curve, auc, RocCurveDisplay, precision_recall_curve, average_precision_score, precision_score, f1_score, recall_score, confusion_matrix
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud") # Path to the directory
```

</details>

<details>
<summary>Original cell 6</summary>

```text
"""
data = pd.read_csv(path) <--- doesn't work since the path is an entire directory, not a csv file.
"""

file_list = os.listdir(path) # the list of files inside the directory
print(file_list) # Showing the files in the list
for file in file_list:
  if file.endswith(".csv"):
    df = pd.read_csv(os.path.join(path,file)) # os.path.join create the complete path to the data file
```

</details>

<details>
<summary>Original cell 11</summary>

```text
#@title Showing the correlation between features
df.corr().style.background_gradient(cmap='coolwarm')
```

</details>

<details>
<summary>Original cell 12</summary>

```text
#@title Pie Chart
import matplotlib.pyplot as plt

# Generate the satisfaction data
satisfaction = df['Class'].value_counts()

# Plot the pie chart
plt.figure(figsize=(8, 4))  # Adjust the figure size as needed
plt.pie(
    satisfaction,
    labels=['Non-Fraud', 'Fraud'],
    autopct='%1.1f%%'
)
plt.title('Class')
plt.show()
```

</details>

<details>
<summary>Original cell 14</summary>

```text
df.drop_duplicates(inplace=True)
df.info()
```

</details>

<details>
<summary>Original cell 15</summary>

```text
"""
No missing values found.
No preprocessing done to features V1->V28 since they are already scaled.
The only dilemma is the Imbalance.
'Amount' column is highly right-skewed
"""
```

</details>

<details>
<summary>Original cell 17</summary>

```text
#@title Splitting into x and y
x = df.drop('Class', axis=1)
y = df['Class']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)
```

</details>

<details>
<summary>Original cell 18</summary>

```text
#@title Scaling the 'Time' Column
scaler = RobustScaler()
x_train['Time'] = scaler.fit_transform(x_train['Time'].values.reshape(-1, 1))
x_test['Time'] = scaler.fit_transform(x_test['Time'].values.reshape(-1, 1))
```

</details>

<details>
<summary>Original cell 19</summary>

```text
#@title Scaling the 'Amount' Column
x_train['Amount'] = scaler.fit_transform(x_train['Amount'].values.reshape(-1, 1))
x_test['Amount'] = scaler.fit_transform(x_test['Amount'].values.reshape(-1, 1))
x_train.describe()
```

</details>

<details>
<summary>Original cell 21</summary>

```text
#@title Handling Class Imbalance
over_sampling = SMOTE(random_state=42)
x_train,y_train = over_sampling.fit_resample(x_train,y_train)

y_train.value_counts()
```

</details>

<details>
<summary>Original cell 22</summary>

```text
Keep in mind that for this particular credit card fraud dataset, `RobustScaler` is often preferred for the 'Amount' column because it's less sensitive to outliers, which are common in transaction amounts, and the values can sometimes be zero, which `log` transformation can't handle directly without adding a constant.
```

</details>

<details>
<summary>Original cell 24</summary>

```text
from sklearn.metrics import log_loss

LR_model = LogisticRegression(class_weight= 'balanced') # initializing the model
LR_model.fit(x_train, y_train) # training the model
loss = log_loss(y_test, LR_model.predict_proba(x_test)) # y_test vs y_predicted
print(f"Logg loss: {loss:.4f}")
proba = LR_model.predict_proba(x_test) #  used to get the probability for each data point belonging to each class

y_pred = LR_model.predict(x_test)

"You can either perform oversampling/undersampling or you can just set up the weights for each class, You have the choice"
```

</details>

<details>
<summary>Original cell 25</summary>

```text
#@title ROC Curve
from sklearn.metrics import roc_curve, auc, RocCurveDisplay
import matplotlib.pyplot as plt

# Use actual y_test and predicted probabilities from your logistic regression model
y_true = y_test
y_pred_proba = proba[:, 1]  # Probabilities for the positive class (Class=1)

# Compute ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
roc_auc = auc(fpr, tpr)

# Plot the ROC curve using RocCurveDisplay for a cleaner plot
plt.figure(figsize=(8, 6))
RocCurveDisplay.from_predictions(y_true, y_pred_proba)
plt.title('Receiver Operating Characteristic (ROC) Curve for Logistic Regression')
plt.grid(True)
plt.show()
```

</details>

<details>
<summary>Original cell 26</summary>

```text
#@title Precision-Recall Curve
from sklearn.metrics import precision_recall_curve, average_precision_score, PrecisionRecallDisplay
import matplotlib.pyplot as plt

# Use actual y_test and predicted probabilities from your logistic regression model
y_true = y_test
y_pred_proba = proba[:, 1]  # Probabilities for the positive class (Class=1)

# Compute precision-recall curve and average precision score
precision, recall, _ = precision_recall_curve(y_true, y_pred_proba)
# average_precision = average_precision_score(y_true, y_pred_proba)

# Plot the Precision-Recall curve using PrecisionRecallDisplay
plt.figure(figsize=(8, 6))
PrecisionRecallDisplay.from_predictions(y_true, y_pred_proba)
plt.title('Precision-Recall Curve for Logistic Regression')
plt.grid(True)
plt.show()
```

</details>